# LLM Provider Fusion ML Ensembles

Ноутбук собирает provider-level ensemble для ML-моделей на LLM-признаках. Для каждого провайдера и семейства схем выбирается лучший вариант, после чего строятся одиночные модели и ансамбли.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import ast
import json
import time
from datetime import datetime
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from scipy.optimize import nnls, minimize

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.metrics import pairwise_distances
from sklearn.pipeline import Pipeline

from sklearn.linear_model import (
    Ridge,
    Lasso,
    ElasticNet,
    BayesianRidge,
    HuberRegressor,
    LinearRegression,
    QuantileRegressor,
)
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

seed = 2
np.random.seed(seed)

In [ ]:
# ------------------------------------------------------------
# Global config
# ------------------------------------------------------------
target_col = "duration_hours"
cap = 960


def require_path(env_name: str, label: str) -> Path:
    raw_value = os.environ.get(env_name, "").strip()
    if not raw_value:
        raise FileNotFoundError(f"Укажи путь к {label} через переменную окружения {env_name}.")
    path = Path(raw_value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


DATA_PATH = require_path("DATA_FINALL_PATH", "data_finall")
ARTIFACT_ROOT = Path(
    os.environ.get("LLM_PROVIDER_FUSION_ML_ARTIFACTS_DIR", "./artifacts_llm_provider_fusion_ensemble")
).expanduser()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = ARTIFACT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

PROVIDER_MANUAL_ARTIFACT_DIRS = {
    "gpt": os.environ.get("GPT_LLM_ARTIFACT_DIR", "./artifacts"),
    "claude": os.environ.get("CLAUDE_LLM_ARTIFACT_DIR", "./claude_api/artifacts_claude_api"),
    "ollama": os.environ.get("OLLAMA_LLM_ARTIFACT_DIR", "./ollama_local/artifacts_ollama_local_pristine"),
    "qwen": os.environ.get("QWEN_LLM_ARTIFACT_DIR", "./qwen_local/artifacts_qwen_local_pristine"),
}

PROVIDER_ENV_DIR_VARS = {
    "gpt": "GPT_LLM_ARTIFACT_DIR",
    "claude": "CLAUDE_LLM_ARTIFACT_DIR",
    "ollama": "OLLAMA_LLM_ARTIFACT_DIR",
    "qwen": "QWEN_LLM_ARTIFACT_DIR",
}

KNOWN_ARTIFACT_DIR_NAMES = {
    "gpt": ["artifacts", "artifacts_gpt", "artifacts_gpt_api", "artifacts_gpt_pristine"],
    "claude": ["artifacts_claude_api"],
    "ollama": ["artifacts_ollama_local_pristine", "artifacts_ollama_local"],
    "qwen": ["artifacts_qwen_local_pristine", "artifacts_qwen_local"],
}

SEARCH_ROOTS = [Path(".")]

FEATURE_FILE_PATTERNS = {
    "gpt": ["llm_features_{variant}_{split}.csv", "llm_features_gpt*_{variant}_{split}.csv"],
    "claude": ["llm_features_claude_api*_{variant}_{split}.csv"],
    "ollama": ["llm_features_ollama*_{variant}_{split}.csv"],
    "qwen": ["llm_features_qwen_local*_{variant}_{split}.csv"],
}

SCHEME_FAMILY_ORDER = ["llm_only", "cluster_before_llm", "llm_then_cluster"]

FIXED_CLUSTER_CONFIGS = [
    {"tag": "gmm_diag_5", "clusterer": "GaussianMixture", "params": {"n_components": 5, "covariance_type": "diag"}},
    {"tag": "mbkm_2", "clusterer": "MiniBatchKMeans", "params": {"n_clusters": 2}},
]

MODEL_INCLUDE = None
KEEP_ONLY_POSITIVE_DELTA = False
MAX_BASE_LEARNERS = None

BLEND_TRAIN_FRAC = 0.85
BLEND_FIT_FRAC = 0.50

RUN_ALL_PAIRS = True
RUN_ALL_TRIPLES = True
RUN_TOPK_PREFIX_ENSEMBLES = True
RUN_GREEDY_SEARCH = True
RUN_STACKERS = True
RUN_EXHAUSTIVE_TOPN_SUBSETS = True
EXHAUSTIVE_TOP_N = 10
REFIT_TOP_ENSEMBLES = 300
PAIR_WEIGHT_GRID = np.linspace(0.0, 1.0, 101)

FORCE_REFRESH_BASE_PREDICTIONS = False
RESUME_IF_PREDICTIONS_EXIST = True


In [ ]:
# ------------------------------------------------------------
# Save helpers
# ------------------------------------------------------------
def save_df(df: pd.DataFrame, name: str):
    run_path = RUN_DIR / name
    latest_path = ARTIFACT_ROOT / name.replace(".csv", "_latest.csv")
    df.to_csv(run_path, index=False)
    df.to_csv(latest_path, index=False)
    return run_path, latest_path

def save_json(payload, name: str):
    run_path = RUN_DIR / name
    latest_path = ARTIFACT_ROOT / name.replace(".json", "_latest.json")
    run_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    latest_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return run_path, latest_path

def sanitize_name(name: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in str(name))

def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)

def mae_metric(y, p):
    return float(mean_absolute_error(y, clip_pred(p)))

def rmse_metric(y, p):
    return float(np.sqrt(mean_squared_error(y, clip_pred(p))))

def mdae_metric(y, p):
    return float(median_absolute_error(y, clip_pred(p)))

def score_predictions(y_true, y_pred):
    return {
        "MAE": mae_metric(y_true, y_pred),
        "RMSE": rmse_metric(y_true, y_pred),
        "MdAE": mdae_metric(y_true, y_pred),
    }

In [ ]:
# ------------------------------------------------------------
# Provider run discovery
# ------------------------------------------------------------
def _normalize_candidate_path(raw):
    if raw is None:
        return None
    p = Path(raw)
    if p.is_dir():
        return p / "llm_cluster_compare_results.csv"
    return p

def _dedupe_paths(paths):
    seen = set()
    out = []
    for p in paths:
        try:
            rp = str(Path(p).resolve())
        except Exception:
            rp = str(Path(p))
        if rp not in seen:
            out.append(Path(p))
            seen.add(rp)
    return out

def _candidate_sort_key(path: Path, provider: str):
    try:
        mtime = -path.stat().st_mtime
    except Exception:
        mtime = 0.0
    parent_name = path.parent.name
    expected = KNOWN_ARTIFACT_DIR_NAMES.get(provider, [])
    exact_bonus = 0 if parent_name in expected else 1
    return (exact_bonus, mtime, len(str(path)), str(path))

def find_provider_summary_candidates(provider: str):
    candidates = []

    manual = PROVIDER_MANUAL_ARTIFACT_DIRS.get(provider)
    if manual:
        p = _normalize_candidate_path(manual)
        if p and p.exists():
            candidates.append(p)

    env_var = PROVIDER_ENV_DIR_VARS.get(provider)
    if env_var:
        env_value = os.getenv(env_var)
        if env_value:
            p = _normalize_candidate_path(env_value)
            if p and p.exists():
                candidates.append(p)

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for dir_name in KNOWN_ARTIFACT_DIR_NAMES.get(provider, []):
            pattern = f"**/{dir_name}/llm_cluster_compare_results.csv"
            try:
                candidates.extend(root.glob(pattern))
            except Exception:
                pass

    candidates = [Path(p) for p in candidates if Path(p).exists()]
    candidates = _dedupe_paths(candidates)
    candidates = sorted(candidates, key=lambda p: _candidate_sort_key(p, provider))
    return candidates

def resolve_provider_registry():
    rows = []
    registry = {}
    for provider in ["gpt", "claude", "ollama", "qwen"]:
        cands = find_provider_summary_candidates(provider)
        summary_path = cands[0] if cands else None
        artifact_dir = summary_path.parent if summary_path is not None else None

        rows.append({
            "provider": provider,
            "status": "found" if summary_path is not None else "missing",
            "artifact_dir": str(artifact_dir) if artifact_dir is not None else "",
            "summary_path": str(summary_path) if summary_path is not None else "",
            "n_candidates_found": len(cands),
            "all_candidates": json.dumps([str(p) for p in cands], ensure_ascii=False),
        })

        if summary_path is not None:
            registry[provider] = {
                "summary_path": Path(summary_path),
                "artifact_dir": Path(artifact_dir),
                "all_candidates": cands,
            }

    reg_df = pd.DataFrame(rows)
    return registry, reg_df

PROVIDER_REGISTRY, provider_registry_df = resolve_provider_registry()
display(provider_registry_df)

if len(PROVIDER_REGISTRY) == 0:
    raise FileNotFoundError(
        "Не найден ни один provider artifact dir. Заполни PROVIDER_MANUAL_ARTIFACT_DIRS."
    )

print("Resolved providers:", list(PROVIDER_REGISTRY.keys()))
save_df(provider_registry_df, "provider_registry.csv")

In [ ]:
# ------------------------------------------------------------
# Load data and reproduce the canonical split
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy()
test_typical = test_full[test_full[target_col] < cap].copy()

train_filt = train_full[train_full[target_col] < cap].copy()

meta_Xtr0 = train_filt.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_ytr0 = train_filt[target_col].reset_index(drop=True)

meta_Xte0 = test_full.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte0 = test_full[target_col].reset_index(drop=True)

meta_Xte_typ0 = test_typical.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte_typ0 = test_typical[target_col].reset_index(drop=True)

llm_train_raw = train_filt.reset_index(drop=True).copy()
llm_test_raw = test_full.reset_index(drop=True).copy()
test_typical_mask = (llm_test_raw[target_col] < cap).reset_index(drop=True)

print("train_full   :", train_full.shape)
print("train_filt   :", train_filt.shape)
print("test_full    :", test_full.shape)
print("test_typical :", test_typical.shape)
print("meta_Xtr0    :", meta_Xtr0.shape)
print("meta_Xte0    :", meta_Xte0.shape)
print("meta_Xte_typ0:", meta_Xte_typ0.shape)

In [ ]:
# ------------------------------------------------------------
# Cluster helpers and raw-space cluster variants
# ------------------------------------------------------------
def make_clusterer(name, params):
    if name == "MiniBatchKMeans":
        return MiniBatchKMeans(
            n_clusters=params["n_clusters"],
            random_state=seed,
            n_init=20,
            batch_size=1024,
        )
    elif name == "GaussianMixture":
        return GaussianMixture(
            n_components=params["n_components"],
            covariance_type=params["covariance_type"],
            random_state=seed,
        )
    else:
        raise ValueError(name)

def probs_from_dist(all_d):
    inv = 1.0 / (all_d + 1e-6)
    return inv / inv.sum(axis=1, keepdims=True)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    est = make_clusterer(name, params)

    if name == "MiniBatchKMeans":
        est.fit(Xtr)
        tr_labels = est.predict(Xtr).astype(int)
        te_labels = est.predict(Xte).astype(int)

        if len(np.unique(tr_labels)) < 2:
            return None

        tr_d = pairwise_distances(Xtr, est.cluster_centers_)
        te_d = pairwise_distances(Xte, est.cluster_centers_)
        tr_proba = probs_from_dist(tr_d)
        te_proba = probs_from_dist(te_d)
        return tr_labels, te_labels, tr_proba, te_proba, "native"

    if name == "GaussianMixture":
        est.fit(Xtr)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)

        tr_labels = np.argmax(tr_proba, axis=1).astype(int)
        te_labels = np.argmax(te_proba, axis=1).astype(int)

        if len(np.unique(tr_labels)) < 2:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, "native"

    return None

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({"cluster_id": tr_labels})
    te_feat = pd.DataFrame({"cluster_id": te_labels})

    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat["cluster_size_train"] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat["cluster_size_train"] = pd.Series(te_labels).map(tr_sizes).fillna(0)

    tr_feat["is_noise"] = (tr_feat["cluster_id"] == -1).astype(int)
    te_feat["is_noise"] = (te_feat["cluster_id"] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat["cluster_id"], prefix="cluster")
    te_ohe = pd.get_dummies(te_feat["cluster_id"], prefix="cluster").reindex(columns=tr_ohe.columns, fill_value=0)

    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        for k in range(tr_proba.shape[1]):
            tr_feat[f"cluster_proba_{k}"] = tr_proba[:, k]
            te_feat[f"cluster_proba_{k}"] = te_proba[:, k]

    return tr_feat, te_feat

raw_cluster_scaler = StandardScaler()
raw_Xtr_scaled = raw_cluster_scaler.fit_transform(meta_Xtr0)
raw_Xte_scaled = raw_cluster_scaler.transform(meta_Xte0)

raw_cluster_pca = PCA(n_components=min(30, meta_Xtr0.shape[1]), random_state=seed)
raw_Xtr_embed = raw_cluster_pca.fit_transform(raw_Xtr_scaled)
raw_Xte_embed = raw_cluster_pca.transform(raw_Xte_scaled)

raw_cluster_variants = {}
raw_cluster_summary_rows = []

for cfg in FIXED_CLUSTER_CONFIGS:
    result = fit_clusterer_and_assign(cfg["clusterer"], cfg["params"], raw_Xtr_embed, raw_Xte_embed)
    if result is None:
        raise RuntimeError(f"Ошибка raw cluster config: {cfg}")

    tr_labels, te_labels, tr_proba, te_proba, fit_mode = result
    tr_feat, te_feat = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)

    raw_cluster_variants[cfg["tag"]] = {
        "cfg": cfg,
        "fit_mode": fit_mode,
        "tr_labels": tr_labels,
        "te_labels": te_labels,
        "tr_proba": tr_proba,
        "te_proba": te_proba,
        "train_feat_raw": tr_feat.reset_index(drop=True),
        "test_feat_raw": te_feat.reset_index(drop=True),
    }

    raw_cluster_summary_rows.append({
        "tag": cfg["tag"],
        "clusterer": cfg["clusterer"],
        "params": json.dumps(cfg["params"], ensure_ascii=False),
        "fit_mode": fit_mode,
        "n_clusters_train": int(len(np.unique(tr_labels))),
        "train_rows": int(len(tr_labels)),
        "test_rows": int(len(te_labels)),
    })

raw_cluster_summary_df = pd.DataFrame(raw_cluster_summary_rows)
display(raw_cluster_summary_df)
save_df(raw_cluster_summary_df, "raw_cluster_summary.csv")

In [ ]:
# ------------------------------------------------------------
# Load provider-level comparison tables and apply the 12-candidate logic
# ------------------------------------------------------------
def experiment_to_scheme_family(experiment_name: str):
    if experiment_name == "llm_only":
        return "llm_only"
    if str(experiment_name).startswith("cluster_before_llm__"):
        return "cluster_before_llm"
    if str(experiment_name).startswith("llm_then_cluster__"):
        return "llm_then_cluster"
    return None

def experiment_to_cluster_tag(experiment_name: str):
    if experiment_name == "llm_only":
        return None
    if "__" in str(experiment_name):
        return str(experiment_name).split("__", 1)[1]
    return None

required_cols = [
    "experiment",
    "model",
    "cv_MAE_internal",
    "holdout_typical_MAE",
    "holdout_full_MAE",
    "best_params",
]

provider_frames = []
for provider, info in PROVIDER_REGISTRY.items():
    part = pd.read_csv(info["summary_path"])

    # defensive compatibility
    if "cv_MAE_internal" not in part.columns and "cv_MAE" in part.columns:
        part = part.rename(columns={"cv_MAE": "cv_MAE_internal"})

    missing = [c for c in required_cols if c not in part.columns]
    if missing:
        raise ValueError(f"{provider}: missing columns in {info['summary_path']}: {missing}")

    part["provider"] = provider
    part["artifact_dir"] = str(info["artifact_dir"])
    part["scheme_family"] = part["experiment"].map(experiment_to_scheme_family)
    part["cluster_tag"] = part["experiment"].map(experiment_to_cluster_tag)

    if MODEL_INCLUDE is not None:
        part = part[part["model"].isin(MODEL_INCLUDE)].copy()

    provider_frames.append(part)

all_provider_rows = pd.concat(provider_frames, ignore_index=True)
all_provider_rows = all_provider_rows[all_provider_rows["scheme_family"].notna()].copy()

sort_cols = ["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE", "provider", "scheme_family", "experiment", "model"]
all_provider_rows = all_provider_rows.sort_values(sort_cols, kind="stable").reset_index(drop=True)

provider_family_model_best_df = (
    all_provider_rows
    .groupby(["provider", "scheme_family", "model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

selected_best12_per_model_df = (
    provider_family_model_best_df
    .sort_values(sort_cols, kind="stable")
    .groupby("model", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

provider_family_model_best_df["provider_family_id"] = (
    provider_family_model_best_df["provider"] + "::" + provider_family_model_best_df["scheme_family"]
)
selected_best12_per_model_df["provider_family_id"] = (
    selected_best12_per_model_df["provider"] + "::" + selected_best12_per_model_df["scheme_family"]
)
selected_best12_per_model_df["base_id"] = (
    selected_best12_per_model_df["provider"]
    + "::"
    + selected_best12_per_model_df["scheme_family"]
    + "::"
    + selected_best12_per_model_df["experiment"]
    + "::"
    + selected_best12_per_model_df["model"]
)

print("All provider rows:", len(all_provider_rows))
print("Best row per (provider, family, model):", len(provider_family_model_best_df))
print("Final selected base learners (best of up to 12 per model):", len(selected_best12_per_model_df))

display(
    provider_family_model_best_df[
        ["provider", "scheme_family", "experiment", "model", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ]
    .sort_values(["model", "cv_MAE_internal", "provider", "scheme_family"], kind="stable")
    .reset_index(drop=True)
)

display(
    selected_best12_per_model_df[
        ["base_id", "model", "provider", "scheme_family", "experiment", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ]
    .sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"], kind="stable")
    .reset_index(drop=True)
)

save_df(all_provider_rows, "provider_compare_all_rows.csv")
save_df(provider_family_model_best_df, "provider_family_model_best.csv")
save_df(selected_best12_per_model_df, "selected_best12_per_model.csv")

selection_payload = {
    "selection_level_1": "within each (provider, scheme_family, model) choose the best low-level experiment by cv_MAE_internal",
    "selection_level_2": "for each ML model choose the best candidate across provider x scheme_family (up to 12 candidates)",
    "providers_found": list(PROVIDER_REGISTRY.keys()),
    "n_all_rows": int(len(all_provider_rows)),
    "n_provider_family_model_best_rows": int(len(provider_family_model_best_df)),
    "n_final_base_learners": int(len(selected_best12_per_model_df)),
    "final_base_learners": selected_best12_per_model_df["base_id"].tolist(),
}
save_json(selection_payload, "selection_logic_summary.json")

## Rebuild exact feature spaces for the selected provider/scheme winners

Ниже notebook не дергает LLM API заново.

Он переиспользует уже сохраненные `llm_features_*.csv` из провайдерских прогонов и заново собирает только нужные design matrices:

- `llm_only`
- `cluster_before_llm__{tag}`
- `llm_then_cluster__{tag}`

Причем для `llm_then_cluster` кластеризация после LLM-фич заново воспроизводится **на основе feature csv конкретного провайдера**, чтобы пространство признаков было тем же по логике, что и в исходных provider notebooks.

In [ ]:
# ------------------------------------------------------------
# Feature-file resolver and provider bundle reconstruction
# ------------------------------------------------------------
PROVIDER_BUNDLE_CACHE = {}
PROVIDER_FEATURE_PATH_LOG = {}

def normalize_llm_feature_frame(df_part: pd.DataFrame):
    if "row_id" not in df_part.columns:
        raise ValueError("feature csv must contain row_id")

    out = df_part.sort_values("row_id").reset_index(drop=True).copy()
    out = out.set_index("row_id")

    if "parse_error" not in out.columns:
        out["parse_error"] = 0

    numeric_cols = [c for c in out.columns if c != "parse_error"]
    for c in numeric_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    medians = out[numeric_cols].median(numeric_only=True)
    out[numeric_cols] = out[numeric_cols].fillna(medians)
    out["parse_error"] = out["parse_error"].fillna(0).astype(int)
    return out

def feature_file_candidates_for_provider(provider: str, variant_name: str, split_name: str):
    if provider not in PROVIDER_REGISTRY:
        raise KeyError(provider)

    art_dir = PROVIDER_REGISTRY[provider]["artifact_dir"]
    patterns = [
        pat.format(variant=variant_name, split=split_name)
        for pat in FEATURE_FILE_PATTERNS[provider]
    ]

    matches = []
    for pat in patterns:
        matches.extend(sorted(art_dir.glob(pat)))

    matches = _dedupe_paths([p for p in matches if p.exists()])
    matches = sorted(
        matches,
        key=lambda p: (len(p.name), -p.stat().st_mtime if p.exists() else 0.0, str(p)),
    )
    return matches, patterns

def load_provider_variant_features(provider: str, variant_name: str, split_name: str):
    matches, patterns = feature_file_candidates_for_provider(provider, variant_name, split_name)
    if not matches:
        art_dir = PROVIDER_REGISTRY[provider]["artifact_dir"]
        sample = sorted([p.name for p in art_dir.glob("llm_features*.csv")])[:25]
        raise FileNotFoundError(
            f"{provider}: feature file not found for variant={variant_name}, split={split_name}. "
            f"Patterns={patterns}. artifact_dir={art_dir}. Sample existing files={sample}"
        )

    path = matches[0]
    frame = normalize_llm_feature_frame(pd.read_csv(path))

    expected_len = len(llm_train_raw) if split_name == "train" else len(llm_test_raw)
    if len(frame) != expected_len:
        raise ValueError(
            f"{provider}: unexpected length for {path.name}. "
            f"expected={expected_len}, got={len(frame)}"
        )

    return frame, path

def build_provider_feature_bundles(provider: str):
    if provider in PROVIDER_BUNDLE_CACHE:
        return PROVIDER_BUNDLE_CACHE[provider]

    bundles = {}
    path_rows = []

    # 1) llm_only
    llm_train_feat_only, path_train_only = load_provider_variant_features(provider, "llm_only", "train")
    llm_test_feat_only, path_test_only = load_provider_variant_features(provider, "llm_only", "test")

    Xtr_llm_only = pd.concat(
        [meta_Xtr0.reset_index(drop=True), llm_train_feat_only.reset_index(drop=True)],
        axis=1,
    )
    Xte_llm_only_full = pd.concat(
        [meta_Xte0.reset_index(drop=True), llm_test_feat_only.reset_index(drop=True)],
        axis=1,
    )
    Xte_llm_only_typ = Xte_llm_only_full.loc[test_typical_mask].reset_index(drop=True)

    bundles["llm_only"] = {
        "Xtr": Xtr_llm_only,
        "Xte_full": Xte_llm_only_full,
        "Xte_typ": Xte_llm_only_typ,
        "feature_sources": {
            "llm_train": str(path_train_only),
            "llm_test": str(path_test_only),
        },
    }

    path_rows.extend([
        {"provider": provider, "experiment": "llm_only", "split": "train", "path": str(path_train_only)},
        {"provider": provider, "experiment": "llm_only", "split": "test", "path": str(path_test_only)},
    ])

    # 2) cluster_before_llm variants
    for tag, raw_bundle in raw_cluster_variants.items():
        exp_name = f"cluster_before_llm__{tag}"
        llm_train_feat, path_train = load_provider_variant_features(provider, exp_name, "train")
        llm_test_feat, path_test = load_provider_variant_features(provider, exp_name, "test")

        Xtr = pd.concat(
            [
                meta_Xtr0.reset_index(drop=True),
                raw_bundle["train_feat_raw"].reset_index(drop=True),
                llm_train_feat.reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_full = pd.concat(
            [
                meta_Xte0.reset_index(drop=True),
                raw_bundle["test_feat_raw"].reset_index(drop=True),
                llm_test_feat.reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_typ = Xte_full.loc[test_typical_mask].reset_index(drop=True)

        bundles[exp_name] = {
            "Xtr": Xtr,
            "Xte_full": Xte_full,
            "Xte_typ": Xte_typ,
            "feature_sources": {
                "llm_train": str(path_train),
                "llm_test": str(path_test),
            },
        }

        path_rows.extend([
            {"provider": provider, "experiment": exp_name, "split": "train", "path": str(path_train)},
            {"provider": provider, "experiment": exp_name, "split": "test", "path": str(path_test)},
        ])

    # 3) llm_then_cluster variants
    Xtr_llm_space = Xtr_llm_only.copy()
    Xte_llm_space = Xte_llm_only_full.copy()

    cluster_after_scaler = StandardScaler()
    Xtr_llm_scaled = cluster_after_scaler.fit_transform(Xtr_llm_space)
    Xte_llm_scaled = cluster_after_scaler.transform(Xte_llm_space)

    cluster_after_pca = PCA(n_components=min(30, Xtr_llm_space.shape[1]), random_state=seed)
    Xtr_llm_embed = cluster_after_pca.fit_transform(Xtr_llm_scaled)
    Xte_llm_embed = cluster_after_pca.transform(Xte_llm_scaled)

    for cfg in FIXED_CLUSTER_CONFIGS:
        result = fit_clusterer_and_assign(cfg["clusterer"], cfg["params"], Xtr_llm_embed, Xte_llm_embed)
        if result is None:
            raise RuntimeError(f"{provider}: ошибка cluster-after-LLM config: {cfg}")

        aft_tr_labels, aft_te_labels, aft_tr_proba, aft_te_proba, aft_fit_mode = result
        cluster_train_feat, cluster_test_feat = build_cluster_meta_features(
            aft_tr_labels,
            aft_te_labels,
            aft_tr_proba,
            aft_te_proba,
        )

        exp_name = f"llm_then_cluster__{cfg['tag']}"

        Xtr = pd.concat(
            [
                meta_Xtr0.reset_index(drop=True),
                llm_train_feat_only.reset_index(drop=True),
                cluster_train_feat.reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_full = pd.concat(
            [
                meta_Xte0.reset_index(drop=True),
                llm_test_feat_only.reset_index(drop=True),
                cluster_test_feat.reset_index(drop=True),
            ],
            axis=1,
        )
        Xte_typ = Xte_full.loc[test_typical_mask].reset_index(drop=True)

        bundles[exp_name] = {
            "Xtr": Xtr,
            "Xte_full": Xte_full,
            "Xte_typ": Xte_typ,
            "feature_sources": {
                "llm_train": str(path_train_only),
                "llm_test": str(path_test_only),
                "cluster_after_fit_mode": aft_fit_mode,
            },
        }

    PROVIDER_BUNDLE_CACHE[provider] = bundles
    PROVIDER_FEATURE_PATH_LOG[provider] = pd.DataFrame(path_rows).drop_duplicates().reset_index(drop=True)
    return bundles

# smoke-check only the providers actually used by the final selected pool
selected_providers = sorted(selected_best12_per_model_df["provider"].unique().tolist())
bundle_manifest_rows = []
for provider in selected_providers:
    provider_bundles = build_provider_feature_bundles(provider)
    for exp_name, bundle in provider_bundles.items():
        bundle_manifest_rows.append({
            "provider": provider,
            "experiment": exp_name,
            "Xtr_shape": str(bundle["Xtr"].shape),
            "Xte_full_shape": str(bundle["Xte_full"].shape),
            "Xte_typ_shape": str(bundle["Xte_typ"].shape),
            "feature_sources": json.dumps(bundle.get("feature_sources", {}), ensure_ascii=False),
        })

bundle_manifest_df = pd.DataFrame(bundle_manifest_rows)
display(bundle_manifest_df.sort_values(["provider", "experiment"], kind="stable").reset_index(drop=True))

save_df(bundle_manifest_df, "provider_bundle_manifest.csv")

feature_path_log_df = pd.concat(
    [PROVIDER_FEATURE_PATH_LOG[p] for p in selected_providers if p in PROVIDER_FEATURE_PATH_LOG],
    ignore_index=True,
)
save_df(feature_path_log_df, "provider_feature_path_log.csv")

## Rebuild the final base pool and cache its predictions

Теперь каждая база ансамбля — это уже **одна конкретная ML-модель с ее лучшим вариантом среди 12 провайдерско-схемных кандидатов**.

То есть base learner имеет вид:

`provider :: scheme_family :: low_level_experiment :: model`

и использует именно те `best_params`, которые были найдены в исходном provider notebook для этой конкретной строки.

In [ ]:
# ------------------------------------------------------------
# Model builders for refit of selected base learners
# ------------------------------------------------------------
MODEL_BUILDERS = {
    "Ridge": lambda: Pipeline([("sc", StandardScaler()), ("m", Ridge(random_state=seed))]),
    "Lasso": lambda: Pipeline([("sc", StandardScaler()), ("m", Lasso(random_state=seed, max_iter=100000))]),
    "ElasticNet": lambda: Pipeline([("sc", StandardScaler()), ("m", ElasticNet(random_state=seed, max_iter=100000))]),
    "HuberReg": lambda: Pipeline([("sc", StandardScaler()), ("m", HuberRegressor(max_iter=10000))]),
    "BayRidge": lambda: Pipeline([("sc", StandardScaler()), ("m", BayesianRidge())]),
    "RF": lambda: Pipeline([("m", RandomForestRegressor(random_state=seed, n_jobs=-1))]),
    "ExtraTrees": lambda: Pipeline([("m", ExtraTreesRegressor(random_state=seed, n_jobs=-1))]),
    "GBoost": lambda: Pipeline([("m", GradientBoostingRegressor(random_state=seed))]),
    "XGB": lambda: Pipeline([
        ("m", XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            random_state=seed,
            tree_method="hist",
            n_jobs=-1,
            verbosity=0,
        ))
    ]),
    "KNN": lambda: Pipeline([("sc", StandardScaler()), ("m", KNeighborsRegressor())]),
}

def extract_core_estimator(estimator):
    return estimator.steps[-1][1] if isinstance(estimator, Pipeline) else estimator

def parse_params_json(raw):
    if isinstance(raw, dict):
        return raw
    if raw is None:
        return {}
    try:
        if pd.isna(raw):
            return {}
    except Exception:
        pass

    s = str(raw).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return {}

    try:
        return json.loads(s)
    except Exception:
        pass

    try:
        return ast.literal_eval(s)
    except Exception:
        pass

    s2 = re.sub(r"\bNaN\b", "None", s, flags=re.IGNORECASE)
    s2 = re.sub(r"\bnull\b", "None", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\btrue\b", "True", s2, flags=re.IGNORECASE)
    s2 = re.sub(r"\bfalse\b", "False", s2, flags=re.IGNORECASE)
    return ast.literal_eval(s2)

def make_selected_base_estimator(model_name: str, params):
    if model_name not in MODEL_BUILDERS:
        raise KeyError(f"Unknown model: {model_name}")

    est = MODEL_BUILDERS[model_name]()
    core = extract_core_estimator(est)
    allowed = set(core.get_params(deep=False).keys())

    parsed = parse_params_json(params)
    clean = {}
    for k, v in parsed.items():
        if k in allowed:
            clean[k] = v.item() if hasattr(v, "item") else v

    if clean:
        core.set_params(**clean)
    return est

PRED_CACHE_DIR = RUN_DIR / "pred_cache"
for sub in ["blend_val", "fullfit_test_typical", "fullfit_test_full"]:
    (PRED_CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)

blend_split_idx = int(len(meta_Xtr0) * BLEND_TRAIN_FRAC)
if blend_split_idx <= 0 or blend_split_idx >= len(meta_Xtr0):
    raise ValueError("Некорректный BLEND_TRAIN_FRAC.")

base_pool_df = selected_best12_per_model_df.copy()
if KEEP_ONLY_POSITIVE_DELTA and "delta_typical" in base_pool_df.columns:
    base_pool_df = base_pool_df[base_pool_df["delta_typical"] > 0].copy()

if MAX_BASE_LEARNERS is not None:
    base_pool_df = (
        base_pool_df
        .sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"], kind="stable")
        .head(int(MAX_BASE_LEARNERS))
        .copy()
    )

base_pool_df["best_params_parsed"] = base_pool_df["best_params"].apply(parse_params_json)

display(
    base_pool_df[
        ["base_id", "model", "provider", "scheme_family", "experiment", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ]
    .sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"], kind="stable")
    .reset_index(drop=True)
)
save_df(base_pool_df, "selected_base_pool.csv")

In [ ]:
# ------------------------------------------------------------
# Refit selected bases and cache their predictions
# ------------------------------------------------------------
def _pred_path(kind: str, base_id: str) -> Path:
    return PRED_CACHE_DIR / kind / f"{sanitize_name(base_id)}.npy"

def _save_pred(kind: str, base_id: str, arr):
    np.save(_pred_path(kind, base_id), np.asarray(arr, dtype=np.float32))

def _load_pred(kind: str, base_id: str):
    path = _pred_path(kind, base_id)
    if not path.exists():
        return None
    return np.load(path)

def _have_all_cached(base_id: str) -> bool:
    needed = [
        _pred_path("blend_val", base_id),
        _pred_path("fullfit_test_typical", base_id),
        _pred_path("fullfit_test_full", base_id),
    ]
    return all(p.exists() for p in needed)

def resolve_experiment_bundle_from_row(row: pd.Series):
    provider = row["provider"]
    experiment_name = row["experiment"]
    bundles = build_provider_feature_bundles(provider)
    if experiment_name not in bundles:
        raise KeyError(f"{provider}: experiment bundle not found: {experiment_name}")
    return bundles[experiment_name]

def fit_and_predict_base_row(row: pd.Series):
    base_id = row["base_id"]
    model_name = row["model"]
    params = row["best_params_parsed"]

    if RESUME_IF_PREDICTIONS_EXIST and not FORCE_REFRESH_BASE_PREDICTIONS and _have_all_cached(base_id):
        return {
            "base_id": base_id,
            "status": "cached",
            "model": model_name,
            "provider": row["provider"],
            "scheme_family": row["scheme_family"],
            "experiment": row["experiment"],
            "elapsed_sec": 0.0,
        }

    bundle = resolve_experiment_bundle_from_row(row)
    Xtr_all = bundle["Xtr"]
    Xte_typ = bundle["Xte_typ"]
    Xte_full = bundle["Xte_full"]

    X_blend_tr = Xtr_all.iloc[:blend_split_idx].copy()
    X_blend_val = Xtr_all.iloc[blend_split_idx:].copy()
    y_blend_tr = meta_ytr0.iloc[:blend_split_idx].reset_index(drop=True)
    y_blend_val_local = meta_ytr0.iloc[blend_split_idx:].reset_index(drop=True)

    t0 = time.time()

    est_split = make_selected_base_estimator(model_name, params)
    est_split.fit(X_blend_tr, y_blend_tr)
    pred_blend_val = clip_pred(est_split.predict(X_blend_val))

    est_full = make_selected_base_estimator(model_name, params)
    est_full.fit(Xtr_all, meta_ytr0)
    pred_test_typ = clip_pred(est_full.predict(Xte_typ))
    pred_test_full = clip_pred(est_full.predict(Xte_full))

    _save_pred("blend_val", base_id, pred_blend_val)
    _save_pred("fullfit_test_typical", base_id, pred_test_typ)
    _save_pred("fullfit_test_full", base_id, pred_test_full)

    return {
        "base_id": base_id,
        "status": "rebuilt",
        "model": model_name,
        "provider": row["provider"],
        "scheme_family": row["scheme_family"],
        "experiment": row["experiment"],
        "elapsed_sec": round(time.time() - t0, 2),
        "blend_val_MAE": mae_metric(y_blend_val_local, pred_blend_val),
        "holdout_typical_MAE": mae_metric(meta_yte_typ0, pred_test_typ),
        "holdout_full_MAE": mae_metric(meta_yte0, pred_test_full),
    }

def build_base_prediction_matrices(pool_frame: pd.DataFrame):
    rows = []
    for _, row in pool_frame.iterrows():
        try:
            out = fit_and_predict_base_row(row)
            rows.append(out)
        except Exception as e:
            rows.append({
                "base_id": row["base_id"],
                "status": "error",
                "model": row["model"],
                "provider": row["provider"],
                "scheme_family": row["scheme_family"],
                "experiment": row["experiment"],
                "error": repr(e),
            })
            print(f"Ошибка: {row['base_id']} -> {e}")
    return pd.DataFrame(rows)

def load_prediction_frame(kind: str, base_ids):
    cols = {}
    for base_id in base_ids:
        arr = _load_pred(kind, base_id)
        if arr is not None:
            cols[base_id] = arr
    return pd.DataFrame(cols)

rebuild_log_df = build_base_prediction_matrices(base_pool_df)
display(rebuild_log_df.sort_values(["status", "holdout_typical_MAE"], kind="stable").reset_index(drop=True))
save_df(rebuild_log_df, "rebuild_log.csv")

selected_base_ids = base_pool_df["base_id"].tolist()
val_pred_df = load_prediction_frame("blend_val", selected_base_ids)
test_typ_fullfit_pred_df = load_prediction_frame("fullfit_test_typical", selected_base_ids)
test_full_fullfit_pred_df = load_prediction_frame("fullfit_test_full", selected_base_ids)

available_base_ids = [
    base_id for base_id in selected_base_ids
    if base_id in val_pred_df.columns
    and base_id in test_typ_fullfit_pred_df.columns
    and base_id in test_full_fullfit_pred_df.columns
]

val_pred_df = val_pred_df[available_base_ids].copy()
test_typ_fullfit_pred_df = test_typ_fullfit_pred_df[available_base_ids].copy()
test_full_fullfit_pred_df = test_full_fullfit_pred_df[available_base_ids].copy()

if len(available_base_ids) < 2:
    raise RuntimeError("Для полноценного ансамблирования нужно хотя бы 2 успешно восстановленных base learners.")

y_blend_val = meta_ytr0.iloc[blend_split_idx:].reset_index(drop=True).to_numpy(dtype=float)
y_test_typ = meta_yte_typ0.to_numpy(dtype=float)
y_test_full = meta_yte0.to_numpy(dtype=float)

val_pred_df.to_csv(RUN_DIR / "base_pred_blend_val.csv", index=False)
test_typ_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_typical.csv", index=False)
test_full_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_full.csv", index=False)

base_meta_df = (
    base_pool_df[
        ["base_id", "model", "provider", "scheme_family", "experiment", "cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ]
    .set_index("base_id")
    .loc[available_base_ids]
    .reset_index()
)

display(base_meta_df)
save_df(base_meta_df, "available_base_meta.csv")

print("Available base learners:", len(available_base_ids))
print(available_base_ids)

## Search over blending / stacking specifications

In [ ]:
# ------------------------------------------------------------
# Ensemble helpers
# ------------------------------------------------------------
def get_base_leaderboard_from_predictions(pred_df: pd.DataFrame, y_true):
    rows = []
    for col in pred_df.columns:
        rows.append({
            "base_id": col,
            "MAE": mae_metric(y_true, pred_df[col].values),
            "RMSE": rmse_metric(y_true, pred_df[col].values),
            "MdAE": mdae_metric(y_true, pred_df[col].values),
        })
    out = pd.DataFrame(rows).sort_values(["MAE", "RMSE", "MdAE"], kind="stable").reset_index(drop=True)
    meta = base_pool_df[["base_id", "model", "provider", "scheme_family", "experiment"]].drop_duplicates()
    return out.merge(meta, on="base_id", how="left")

def aggregate_predictions(pred_matrix: np.ndarray, method: str, trim_frac: float = 0.1):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    if pred_matrix.ndim == 1:
        return clip_pred(pred_matrix)
    if method == "mean":
        out = pred_matrix.mean(axis=1)
    elif method == "median":
        out = np.median(pred_matrix, axis=1)
    elif method == "trimmed_mean":
        m = pred_matrix.shape[1]
        k = int(np.floor(m * trim_frac))
        if k <= 0 or 2 * k >= m:
            out = pred_matrix.mean(axis=1)
        else:
            sorted_mat = np.sort(pred_matrix, axis=1)
            out = sorted_mat[:, k:m-k].mean(axis=1)
    else:
        raise KeyError(method)
    return clip_pred(out)

def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s

def weights_from_errors(pred_fit: pd.DataFrame, y_fit, scheme: str, temperature: float = 1.0, power: float = 1.0):
    errs = np.array([mae_metric(y_fit, pred_fit[c].values) for c in pred_fit.columns], dtype=float)
    if scheme == "equal":
        w = np.ones(len(errs), dtype=float)
    elif scheme == "inverse_mae":
        w = 1.0 / np.maximum(errs, 1e-9)
    elif scheme == "inverse_mae_sq":
        w = 1.0 / np.maximum(errs, 1e-9) ** 2
    elif scheme == "softmax_mae":
        z = -errs / max(float(temperature), 1e-6)
        z = z - z.max()
        w = np.exp(z)
    elif scheme == "rank_power":
        order = np.argsort(errs)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(errs) + 1)
        w = 1.0 / np.power(ranks, power)
    else:
        raise KeyError(scheme)
    return normalize_weights(w)

def fit_weighted_blender(pred_fit: pd.DataFrame, y_fit, scheme: str, **kwargs):
    X = pred_fit.values.astype(float)
    if scheme in {"equal", "inverse_mae", "inverse_mae_sq", "softmax_mae", "rank_power"}:
        w = weights_from_errors(pred_fit, y_fit, scheme, **kwargs)
        return {"kind": "weights", "weights": w}
    elif scheme == "nnls":
        w, _ = nnls(X, np.asarray(y_fit, dtype=float))
        w = normalize_weights(w)
        return {"kind": "weights", "weights": w}
    elif scheme == "simplex_mae":
        n = X.shape[1]
        x0 = np.ones(n, dtype=float) / n
        bounds = [(0.0, 1.0)] * n
        cons = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}

        def objective(w):
            return mae_metric(y_fit, X @ w)

        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=[cons],
            options={"maxiter": 500, "ftol": 1e-8},
        )
        w = normalize_weights(res.x if res.success else x0)
        return {"kind": "weights", "weights": w, "optimizer_success": bool(res.success)}
    else:
        raise KeyError(scheme)

def predict_weighted_blender(fitted, pred_df: pd.DataFrame):
    w = np.asarray(fitted["weights"], dtype=float)
    return clip_pred(pred_df.values @ w)

def build_stacker(stacker_name: str):
    if stacker_name == "linear":
        return LinearRegression()
    elif stacker_name == "positive_linear":
        return LinearRegression(positive=True)
    elif stacker_name == "ridge":
        return Ridge(alpha=1.0)
    elif stacker_name == "lasso":
        return Lasso(alpha=1e-3, max_iter=20000, random_state=seed)
    elif stacker_name == "elastic":
        return ElasticNet(alpha=1e-3, l1_ratio=0.5, max_iter=20000, random_state=seed)
    elif stacker_name == "bayes":
        return BayesianRidge()
    elif stacker_name == "huber":
        return HuberRegressor(max_iter=2000, epsilon=1.35)
    elif stacker_name == "quantile":
        return QuantileRegressor(quantile=0.5, alpha=1e-4, solver="highs")
    elif stacker_name == "gbr":
        return GradientBoostingRegressor(
            loss="absolute_error",
            random_state=seed,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=2,
            min_samples_leaf=5,
            min_samples_split=4,
            subsample=0.9,
        )
    elif stacker_name == "rf":
        return RandomForestRegressor(
            n_estimators=500,
            random_state=seed,
            min_samples_leaf=2,
            n_jobs=-1,
        )
    elif stacker_name == "etr":
        return ExtraTreesRegressor(
            n_estimators=500,
            random_state=seed,
            min_samples_leaf=2,
            n_jobs=-1,
        )
    elif stacker_name == "xgb":
        return XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            n_estimators=500,
            learning_rate=0.03,
            max_depth=2,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.0,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=seed,
            verbosity=0,
        )
    else:
        raise KeyError(stacker_name)

def fit_stacker(pred_fit: pd.DataFrame, y_fit, stacker_name: str):
    model = build_stacker(stacker_name)
    model.fit(pred_fit.values.astype(float), np.asarray(y_fit, dtype=float))
    return {"kind": "stacker", "model": model, "stacker_name": stacker_name}

def predict_stacker(fitted, pred_df: pd.DataFrame):
    return clip_pred(fitted["model"].predict(pred_df.values.astype(float)))

def fit_pair_weight_grid(pred_fit: pd.DataFrame, y_fit):
    if pred_fit.shape[1] != 2:
        raise ValueError("pair_grid expects exactly 2 models")
    best = None
    a = pred_fit.iloc[:, 0].values.astype(float)
    b = pred_fit.iloc[:, 1].values.astype(float)
    for w in PAIR_WEIGHT_GRID:
        pred = clip_pred(w * a + (1.0 - w) * b)
        score = mae_metric(y_fit, pred)
        if best is None or score < best["selection_MAE"]:
            best = {"kind": "pair_grid", "weight_first": float(w), "selection_MAE": score}
    return best

def predict_pair_weight_grid(fitted, pred_df: pd.DataFrame):
    w = float(fitted["weight_first"])
    a = pred_df.iloc[:, 0].values.astype(float)
    b = pred_df.iloc[:, 1].values.astype(float)
    return clip_pred(w * a + (1.0 - w) * b)

def top_models_by_fit_mae(pred_fit_df: pd.DataFrame, y_fit):
    fit_rank_df = get_base_leaderboard_from_predictions(pred_fit_df, y_fit)
    ranked_models = fit_rank_df["base_id"].tolist()
    return ranked_models, fit_rank_df

def spec_tag(spec):
    models_part = "__".join(spec["models"])
    return f'{spec["family"]}::{spec["name"]}::{models_part}'

def evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel):
    models = spec["models"]
    fit_sub = pred_fit_df[models].copy()
    sel_sub = pred_sel_df[models].copy()

    family = spec["family"]
    name = spec["name"]

    if family == "single":
        pred_sel = clip_pred(sel_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_sel = aggregate_predictions(sel_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(fit_sub, y_fit, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_sel = predict_weighted_blender(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", []),
        }
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(fit_sub, y_fit)
        pred_sel = predict_pair_weight_grid(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weight_first": fitted["weight_first"],
        }
    elif family == "stacker":
        fitted = fit_stacker(fit_sub, y_fit, spec["stacker_name"])
        pred_sel = predict_stacker(fitted, sel_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "stacker_name": spec["stacker_name"],
        }
    else:
        raise KeyError(family)

    sel_metrics = score_predictions(y_sel, pred_sel)
    out["selection_MAE"] = sel_metrics["MAE"]
    out["selection_RMSE"] = sel_metrics["RMSE"]
    out["selection_MdAE"] = sel_metrics["MdAE"]
    out["tag"] = spec_tag(spec)
    out["spec"] = spec
    return out

def refit_and_eval_spec(spec, pred_val_df, y_val, pred_test_typ_df, y_test_typ, pred_test_full_df, y_test_full):
    models = spec["models"]
    val_sub = pred_val_df[models].copy()
    test_typ_sub = pred_test_typ_df[models].copy()
    test_full_sub = pred_test_full_df[models].copy()

    family = spec["family"]
    name = spec["name"]

    if family == "single":
        pred_typ = clip_pred(test_typ_sub.iloc[:, 0].values)
        pred_full = clip_pred(test_full_sub.iloc[:, 0].values)
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "aggregate":
        pred_typ = aggregate_predictions(test_typ_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        pred_full = aggregate_predictions(test_full_sub.values, spec["agg_method"], spec.get("trim_frac", 0.1))
        out = {"family": family, "name": name, "models": models, "n_models": len(models)}
    elif family == "weighted":
        fitted = fit_weighted_blender(val_sub, y_val, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_typ = predict_weighted_blender(fitted, test_typ_sub)
        pred_full = predict_weighted_blender(fitted, test_full_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights": fitted.get("weights", []).tolist() if isinstance(fitted.get("weights", []), np.ndarray) else fitted.get("weights", []),
        }
    elif family == "pair_grid":
        fitted = fit_pair_weight_grid(val_sub, y_val)
        pred_typ = predict_pair_weight_grid(fitted, test_typ_sub)
        pred_full = predict_pair_weight_grid(fitted, test_full_sub)
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weight_first": fitted["weight_first"],
            "weights_like": [fitted["weight_first"], 1.0 - fitted["weight_first"]],
        }
    elif family == "stacker":
        fitted = fit_stacker(val_sub, y_val, spec["stacker_name"])
        pred_typ = predict_stacker(fitted, test_typ_sub)
        pred_full = predict_stacker(fitted, test_full_sub)
        weights_like = None
        if hasattr(fitted["model"], "coef_"):
            coef = np.ravel(getattr(fitted["model"], "coef_"))
            if len(coef) == len(models):
                weights_like = coef.tolist()
        out = {
            "family": family,
            "name": name,
            "models": models,
            "n_models": len(models),
            "weights_like": weights_like,
        }
    else:
        raise KeyError(family)

    typ_metrics = score_predictions(y_test_typ, pred_typ)
    full_metrics = score_predictions(y_test_full, pred_full)

    out["tag"] = spec_tag(spec)
    out["MAE_typical"] = typ_metrics["MAE"]
    out["RMSE_typical"] = typ_metrics["RMSE"]
    out["MdAE_typical"] = typ_metrics["MdAE"]
    out["MAE_full"] = full_metrics["MAE"]
    out["RMSE_full"] = full_metrics["RMSE"]
    out["MdAE_full"] = full_metrics["MdAE"]
    return out

def build_greedy_forward_specs(ranked_models, pred_fit_df, y_fit):
    specs = []
    search_modes = [
        ("aggregate", "greedy_mean", {"agg_method": "mean"}),
        ("weighted", "greedy_inverse", {"weight_scheme": "inverse_mae"}),
        ("weighted", "greedy_nnls", {"weight_scheme": "nnls"}),
        ("weighted", "greedy_simplex", {"weight_scheme": "simplex_mae"}),
    ]

    for family, prefix, extra in search_modes:
        current = []
        remaining = ranked_models.copy()

        while remaining:
            best = None
            for m in remaining:
                cand = current + [m]
                fit_sub = pred_fit_df[cand].copy()

                if len(cand) == 1:
                    pred = clip_pred(fit_sub.iloc[:, 0].values)
                elif family == "aggregate":
                    pred = aggregate_predictions(fit_sub.values, extra["agg_method"], extra.get("trim_frac", 0.1))
                elif family == "weighted":
                    fitted = fit_weighted_blender(fit_sub, y_fit, extra["weight_scheme"], **extra.get("weight_kwargs", {}))
                    pred = predict_weighted_blender(fitted, fit_sub)
                else:
                    raise KeyError(family)

                score = mae_metric(y_fit, pred)
                if best is None or score < best["score"]:
                    best = {"models": cand.copy(), "score": score}

            current = best["models"]
            remaining = [m for m in remaining if m not in current]

            if len(current) >= 2:
                spec = {"family": family, "name": f"{prefix}_step{len(current)}", "models": current.copy()}
                spec.update(extra)
                specs.append(spec)

    return specs

In [ ]:
# ------------------------------------------------------------
# Generate ensemble specs and score them on the selection split
# ------------------------------------------------------------
base_leaderboard = get_base_leaderboard_from_predictions(val_pred_df, y_blend_val)
display(base_leaderboard.head(20))
save_df(base_leaderboard, "rebuilt_base_leaderboard.csv")

blend_cut = int(len(y_blend_val) * BLEND_FIT_FRAC)
if blend_cut <= 0 or blend_cut >= len(y_blend_val):
    raise ValueError("Некорректный BLEND_FIT_FRAC.")

pred_fit_df = val_pred_df.iloc[:blend_cut].copy()
pred_sel_df = val_pred_df.iloc[blend_cut:].copy()
y_fit = y_blend_val[:blend_cut].copy()
y_sel = y_blend_val[blend_cut:].copy()

ranked_models, fit_rank_df = top_models_by_fit_mae(pred_fit_df, y_fit)
print("Top base learners by blend-fit MAE:")
display(fit_rank_df.head(20))
save_df(fit_rank_df, "blend_fit_model_ranking.csv")

specs = []

# singles
for m in ranked_models:
    specs.append({"family": "single", "name": "single", "models": [m]})

# all-model global aggregations
all_models = ranked_models.copy()
specs.extend([
    {"family": "aggregate", "name": "all_mean", "models": all_models, "agg_method": "mean"},
    {"family": "aggregate", "name": "all_median", "models": all_models, "agg_method": "median"},
    {"family": "aggregate", "name": "all_trim10", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.10},
    {"family": "aggregate", "name": "all_trim20", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.20},
    {"family": "weighted", "name": "all_inverse_mae", "models": all_models, "weight_scheme": "inverse_mae"},
    {"family": "weighted", "name": "all_inverse_mae_sq", "models": all_models, "weight_scheme": "inverse_mae_sq"},
    {"family": "weighted", "name": "all_softmax_t1", "models": all_models, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}},
    {"family": "weighted", "name": "all_nnls", "models": all_models, "weight_scheme": "nnls"},
    {"family": "weighted", "name": "all_simplex_mae", "models": all_models, "weight_scheme": "simplex_mae"},
])

if RUN_TOPK_PREFIX_ENSEMBLES:
    for k in range(2, len(ranked_models) + 1):
        topk = ranked_models[:k]
        specs.append({"family": "aggregate", "name": f"top{k}_mean", "models": topk, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": f"top{k}_median", "models": topk, "agg_method": "median"})
        if k >= 5:
            specs.append({"family": "aggregate", "name": f"top{k}_trim10", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.10})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae", "models": topk, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae_sq", "models": topk, "weight_scheme": "inverse_mae_sq"})
        for temp in [0.25, 0.5, 1.0, 2.0, 4.0]:
            specs.append({"family": "weighted", "name": f"top{k}_softmax_t{temp}", "models": topk, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": temp}})
        for power in [0.5, 1.0, 2.0]:
            specs.append({"family": "weighted", "name": f"top{k}_rankp{power}", "models": topk, "weight_scheme": "rank_power", "weight_kwargs": {"power": power}})
        specs.append({"family": "weighted", "name": f"top{k}_nnls", "models": topk, "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": f"top{k}_simplex", "models": topk, "weight_scheme": "simplex_mae"})

if RUN_ALL_PAIRS:
    for a, b in combinations(ranked_models, 2):
        specs.append({"family": "aggregate", "name": "pair_mean", "models": [a, b], "agg_method": "mean"})
        specs.append({"family": "pair_grid", "name": "pair_grid", "models": [a, b]})
        specs.append({"family": "weighted", "name": "pair_nnls", "models": [a, b], "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": "pair_simplex", "models": [a, b], "weight_scheme": "simplex_mae"})

if RUN_ALL_TRIPLES:
    for trio in combinations(ranked_models, 3):
        trio = list(trio)
        specs.append({"family": "aggregate", "name": "triple_mean", "models": trio, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": "triple_median", "models": trio, "agg_method": "median"})
        specs.append({"family": "weighted", "name": "triple_inverse", "models": trio, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": "triple_nnls", "models": trio, "weight_scheme": "nnls"})

if RUN_EXHAUSTIVE_TOPN_SUBSETS:
    topn_models = ranked_models[:min(EXHAUSTIVE_TOP_N, len(ranked_models))]
    for r in range(2, len(topn_models) + 1):
        for subset in combinations(topn_models, r):
            subset = list(subset)
            specs.append({"family": "aggregate", "name": f"subset{r}_mean", "models": subset, "agg_method": "mean"})
            specs.append({"family": "aggregate", "name": f"subset{r}_median", "models": subset, "agg_method": "median"})
            if r >= 5:
                specs.append({"family": "aggregate", "name": f"subset{r}_trim10", "models": subset, "agg_method": "trimmed_mean", "trim_frac": 0.10})
            specs.append({"family": "weighted", "name": f"subset{r}_inverse", "models": subset, "weight_scheme": "inverse_mae"})
            specs.append({"family": "weighted", "name": f"subset{r}_inverse_sq", "models": subset, "weight_scheme": "inverse_mae_sq"})
            specs.append({"family": "weighted", "name": f"subset{r}_softmax", "models": subset, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": 1.0}})
            specs.append({"family": "weighted", "name": f"subset{r}_nnls", "models": subset, "weight_scheme": "nnls"})
            specs.append({"family": "weighted", "name": f"subset{r}_simplex", "models": subset, "weight_scheme": "simplex_mae"})

if RUN_GREEDY_SEARCH:
    specs.extend(build_greedy_forward_specs(ranked_models, pred_fit_df, y_fit))

if RUN_STACKERS:
    stackers = ["linear", "positive_linear", "ridge", "lasso", "elastic", "bayes", "huber", "quantile", "gbr", "rf", "etr", "xgb"]
    for stacker_name in stackers:
        specs.append({"family": "stacker", "name": f"all_{stacker_name}", "models": all_models, "stacker_name": stacker_name})
        for k in range(2, len(ranked_models) + 1):
            topk = ranked_models[:k]
            specs.append({"family": "stacker", "name": f"top{k}_{stacker_name}", "models": topk, "stacker_name": stacker_name})

# de-duplicate exact duplicate specs
spec_seen = set()
unique_specs = []
for spec in specs:
    key = json.dumps(
        {
            "family": spec["family"],
            "name": spec["name"],
            "models": spec["models"],
            "agg_method": spec.get("agg_method"),
            "trim_frac": spec.get("trim_frac"),
            "weight_scheme": spec.get("weight_scheme"),
            "weight_kwargs": spec.get("weight_kwargs"),
            "stacker_name": spec.get("stacker_name"),
        },
        sort_keys=True,
        ensure_ascii=False,
    )
    if key not in spec_seen:
        unique_specs.append(spec)
        spec_seen.add(key)

specs = unique_specs
print(f"Total ensemble specs: {len(specs)}")

search_rows = []
for idx, spec in enumerate(specs, start=1):
    if idx % 200 == 0:
        print(f"Selection scoring {idx}/{len(specs)} ...")
    try:
        row = evaluate_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel)
        search_rows.append(row)
    except Exception as e:
        search_rows.append({
            "tag": spec_tag(spec),
            "family": spec["family"],
            "name": spec["name"],
            "n_models": len(spec["models"]),
            "models": spec["models"],
            "selection_MAE": np.inf,
            "selection_RMSE": np.inf,
            "selection_MdAE": np.inf,
            "error": repr(e),
            "spec": spec,
        })

finite_search_rows = [r for r in search_rows if np.isfinite(r["selection_MAE"])]
finite_search_rows = sorted(
    finite_search_rows,
    key=lambda r: (r["selection_MAE"], r["selection_RMSE"], r["selection_MdAE"], r["tag"])
)

search_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "error": r.get("error", ""),
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in finite_search_rows
]).sort_values(["selection_MAE", "selection_RMSE", "selection_MdAE", "tag"], kind="stable").reset_index(drop=True)

display(search_df.head(30))
save_df(search_df, "ensemble_search_leaderboard.csv")

top_refit_rows = finite_search_rows[:min(REFIT_TOP_ENSEMBLES, len(finite_search_rows))]
print(f"Top rows selected for holdout refit: {len(top_refit_rows)}")

In [ ]:
# ------------------------------------------------------------
# Final holdout refit and summary
# ------------------------------------------------------------
final_rows = []
for idx, row in enumerate(top_refit_rows, start=1):
    if idx % 25 == 0:
        print(f"Final holdout refit {idx}/{len(top_refit_rows)} ...")
    spec = row["spec"]
    out = refit_and_eval_spec(
        spec,
        pred_val_df=val_pred_df,
        y_val=y_blend_val,
        pred_test_typ_df=test_typ_fullfit_pred_df,
        y_test_typ=y_test_typ,
        pred_test_full_df=test_full_fullfit_pred_df,
        y_test_full=y_test_full,
    )
    out["selection_MAE"] = row["selection_MAE"]
    out["selection_RMSE"] = row["selection_RMSE"]
    out["selection_MdAE"] = row["selection_MdAE"]
    final_rows.append(out)

final_rows = sorted(
    final_rows,
    key=lambda r: (r["selection_MAE"], r["MAE_typical"], r["MAE_full"], r["tag"])
)

final_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "MAE_typical": r["MAE_typical"],
        "RMSE_typical": r["RMSE_typical"],
        "MdAE_typical": r["MdAE_typical"],
        "MAE_full": r["MAE_full"],
        "RMSE_full": r["RMSE_full"],
        "MdAE_full": r["MdAE_full"],
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weights_like": json.dumps(r.get("weights_like", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in final_rows
]).sort_values(["selection_MAE", "MAE_typical", "MAE_full", "tag"], kind="stable").reset_index(drop=True)

display(final_df.head(30))
save_df(final_df, "ensemble_holdout_leaderboard.csv")

best_single_base_id = base_leaderboard.iloc[0]["base_id"]
single_typ = score_predictions(y_test_typ, test_typ_fullfit_pred_df[best_single_base_id].values)
single_full = score_predictions(y_test_full, test_full_fullfit_pred_df[best_single_base_id].values)

best_ensemble_row = final_rows[0]

comparison_payload = {
    "selection_metric": "blend_select_MAE",
    "blend_train_frac": BLEND_TRAIN_FRAC,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "n_found_providers": int(len(PROVIDER_REGISTRY)),
    "found_providers": list(PROVIDER_REGISTRY.keys()),
    "provider_selection_rule": "for each (provider, scheme_family, model) choose the best low-level experiment by cv_MAE_internal",
    "model_selection_rule": "for each ML model choose the best candidate across provider x scheme_family (up to 12 candidates)",
    "n_candidate_rows": int(len(all_provider_rows)),
    "n_provider_family_model_best_rows": int(len(provider_family_model_best_df)),
    "n_base_learners": int(len(available_base_ids)),
    "available_base_learners": available_base_ids,
    "best_single_base_by_val": best_single_base_id,
    "best_single_base_metrics": {
        "MAE_typical": single_typ["MAE"],
        "RMSE_typical": single_typ["RMSE"],
        "MdAE_typical": single_typ["MdAE"],
        "MAE_full": single_full["MAE"],
        "RMSE_full": single_full["RMSE"],
        "MdAE_full": single_full["MdAE"],
    },
    "best_ensemble": {
        "tag": best_ensemble_row["tag"],
        "family": best_ensemble_row["family"],
        "name": best_ensemble_row["name"],
        "models": best_ensemble_row["models"],
        "selection_MAE": best_ensemble_row["selection_MAE"],
        "MAE_typical": best_ensemble_row["MAE_typical"],
        "RMSE_typical": best_ensemble_row["RMSE_typical"],
        "MdAE_typical": best_ensemble_row["MdAE_typical"],
        "MAE_full": best_ensemble_row["MAE_full"],
        "RMSE_full": best_ensemble_row["RMSE_full"],
        "MdAE_full": best_ensemble_row["MdAE_full"],
        "weights": best_ensemble_row.get("weights", []),
        "weights_like": best_ensemble_row.get("weights_like", []),
        "weight_first": best_ensemble_row.get("weight_first", None),
    },
}

display(pd.DataFrame([
    {"item": "best_single_base_by_val", "value": best_single_base_id},
    {"item": "best_ensemble_tag", "value": best_ensemble_row["tag"]},
    {"item": "best_ensemble_selection_MAE", "value": best_ensemble_row["selection_MAE"]},
    {"item": "best_ensemble_MAE_typical", "value": best_ensemble_row["MAE_typical"]},
    {"item": "best_ensemble_MAE_full", "value": best_ensemble_row["MAE_full"]},
]))

save_json(comparison_payload, "best_ensemble_summary.json")

best_models = best_ensemble_row["models"]
best_weights = best_ensemble_row.get("weights") or best_ensemble_row.get("weights_like") or []
if len(best_weights) == len(best_models) and len(best_weights) > 0:
    wdf = pd.DataFrame({"base_id": best_models, "weight": best_weights})
    wdf = wdf.merge(
        base_pool_df[["base_id", "provider", "scheme_family", "experiment", "model"]],
        on="base_id",
        how="left",
    )
    save_df(wdf, "best_ensemble_weights.csv")

run_manifest = {
    "run_name": RUN_NAME,
    "artifact_root": str(ARTIFACT_ROOT),
    "run_dir": str(RUN_DIR),
    "data_path": str(DATA_PATH),
    "target_col": target_col,
    "duration_cap": cap,
    "blend_train_frac": BLEND_TRAIN_FRAC,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "keep_only_positive_delta": KEEP_ONLY_POSITIVE_DELTA,
    "max_base_learners": MAX_BASE_LEARNERS,
    "selected_base_learners": available_base_ids,
    "n_selected_base_learners": int(len(available_base_ids)),
    "n_search_specs": int(len(specs)),
    "n_refit_ensembles": int(len(top_refit_rows)),
}

save_json(run_manifest, "run_manifest.json")
print("Сохранено:", RUN_DIR.resolve())

In [ ]:
# ------------------------------------------------------------
# Графики
# ------------------------------------------------------------
top_plot = final_df.head(15).copy()
plt.figure(figsize=(12, 6))
plt.barh(top_plot["tag"][::-1], top_plot["MAE_typical"][::-1])
plt.xlabel("MAE_typical")
plt.ylabel("Ensemble")
plt.title("Top-15 provider-fusion LLM+ML ensembles by selection_MAE -> holdout typical")
plt.tight_layout()
plt.show()

top_base_plot = base_leaderboard.head(15).copy()
plt.figure(figsize=(12, 6))
plt.barh(top_base_plot["base_id"][::-1], top_base_plot["MAE"][::-1])
plt.xlabel("Blend-val MAE")
plt.ylabel("Base learner")
plt.title("Top-15 selected base learners on blend validation")
plt.tight_layout()
plt.show()

top_corr_models = ranked_models[:min(10, len(ranked_models))]
corr = val_pred_df[top_corr_models].corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr.values, aspect="auto", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(top_corr_models)), top_corr_models, rotation=90)
plt.yticks(range(len(top_corr_models)), top_corr_models)
plt.title("Correlation of blend-validation predictions")
plt.tight_layout()
plt.show()

corr.to_csv(RUN_DIR / "blend_val_prediction_correlation.csv")
corr.to_csv(ARTIFACT_ROOT / "blend_val_prediction_correlation_latest.csv")

near_ties = final_df[final_df["selection_MAE"] <= final_df["selection_MAE"].min() + 0.5].copy()
display(near_ties.head(30))
display(final_df.head(20))

## Что смотреть после запуска

Главные артефакты этого финального эксперимента:

- `provider_registry.csv` — какие provider artifact dirs реально найдены;
- `provider_compare_all_rows.csv` — все строки из 4 provider notebooks;
- `provider_family_model_best.csv` — лучший low-level вариант внутри каждого `(provider, scheme_family, model)`;
- `selected_best12_per_model.csv` — итоговая выборка: лучший кандидат из `4 × 3 = 12` для каждой ML-модели;
- `selected_base_pool.csv` — финальный base pool перед ансамблированием;
- `rebuild_log.csv` — какие selected bases успешно переобучились и какие метрики дали;
- `rebuilt_base_leaderboard.csv` — ranking base learners на blend-validation;
- `ensemble_search_leaderboard.csv` — ranking всех проверенных blending/stacking схем по selection split;
- `ensemble_holdout_leaderboard.csv` — финальный ranking на внешнем holdout;
- `best_ensemble_summary.json` — финальный победитель;
- `best_ensemble_weights.csv` — веса победителя, если у него есть явные веса;
- `run_manifest.json` — конфигурация конкретного прогона.

Если авто-поиск путей не сработает, достаточно руками указать `PROVIDER_MANUAL_ARTIFACT_DIRS` и перезапустить notebook сверху.